In [1]:
import os
import textwrap
from pathlib import Path
import fitz
from IPython.display import Markdown
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Qdrant
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from llama_parse import LlamaParse
import requests

load_dotenv()

# Load API keys
groq_api_key = os.getenv("GROQ_API_KEY")
llama_parse_key = os.getenv("LLAMA_PARSE")
api_key = os.getenv("WEATHER_API")

In [2]:
def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [3]:
# Load medical research data
pdf_text = read_pdf("Breast_Cancer.pdf")
document_path = Path("data/parsed_document.md")
with document_path.open("w", encoding="utf-8") as f:
    f.write(pdf_text)

In [4]:
# Load live patient metrics (e.g., from IoT devices or simulated sources)
patient_data = {
    "heart_rate": 85,
    "blood_pressure": "120/80",
    "steps_today": 4500,
}

In [5]:
# Process weather data
location = "Coimbatore"
url1 = f"http://api.weatherapi.com/v1/current.json?key={api_key}&q={location}"
response1 = requests.get(url1)
if response1.status_code == 200:
    weather = response1.json()
    temperature = weather['current']['temp_c']
    print(f"Current temperature in {location}: {temperature}°C")
else:
    print(f"Failed to retrieve weather data: {response1.status_code}")
    temperature = None  # Default value in case of failure

Current temperature in Coimbatore: 25.0°C


In [6]:
# Load medical documents for RAG
loader = UnstructuredMarkdownLoader(document_path)
loaded_documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=128)
docs = text_splitter.split_documents(loaded_documents)
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")
qdrant = Qdrant.from_documents(
    docs,
    embeddings,
    path="./db",
    collection_name="medical_embeddings",
)

c:\Users\vishn\Desktop\Programs\CodeOClock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<?, ?it/s]


In [7]:
recommendation_prompt = """
Use the following information to answer the user's question.

Context: {context}
"Details:" : [
Heart Rate: {{heart_rate}}
Blood Pressure: {{blood_pressure}}
Steps Today: {{steps_today}}
Temperature: {{temperature}}
]
Answer the question in JSON format.
Example:
{{
  "recommendations": [
    {{
      "type": "exercise",
      "advice": "Increase daily steps to 7000 for better cardiovascular health."
    }},
    {{
      "type": "diet",
      "advice": "Consume more potassium-rich foods to maintain healthy blood pressure."
    }}
  ]
}}
"""
prompt = PromptTemplate(
    template=recommendation_prompt, 
    input_variables=["context", "heart_rate", "blood_pressure", "steps_today", "temperature"]
)


In [8]:
# Initialize the LLM and RAG pipeline
llm = ChatGroq(temperature=0, model_name="llama-3.1-70b-versatile")
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=qdrant.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": False},
)

In [12]:
# Function to print the response
def print_response(response):
    response_txt = response["result"]
    for chunk in response_txt.split("\n"):
        if not chunk:
            print()
            continue
        print("\n".join(textwrap.wrap(chunk, 100, break_long_words=False)))

In [ ]:
# Execute the RAG pipeline
response = qa.invoke({
    "query": "Generate health recommendations based on the patient's data.",  # Use "question" key
    "heart_rate": patient_data.get("heart_rate"),  # Ensure the key exists
    "blood_pressure": patient_data.get("blood_pressure"),
    "steps_today": patient_data.get("steps_today"),
    "temperature": temperature,
})



{'query': "Generate health recommendations based on the patient's data.", 'heart_rate': 85, 'blood_pressure': '120/80', 'steps_today': 4500, 'temperature': 25.0, 'result': 'It appears that the provided text does not contain any information related to the user\'s question about health recommendations based on details such as heart rate, blood pressure, steps taken today, and temperature. The text seems to be an acknowledgement section from a document about cancer management guidelines.\n\nTo provide a response in the requested JSON format, I would need more context or information about the user\'s question. However, based on the example provided, here is a generic response:\n\n```\n{\n  "recommendations": [\n    {\n      "type": "exercise",\n      "advice": "Increase daily physical activity for better overall health."\n    },\n    {\n      "type": "health",\n      "advice": "Consult a healthcare professional for personalized advice on managing cancer and other health conditions."\n    }

In [14]:
print_response(response)


It appears that the provided text does not contain any information related to the user's question
about health recommendations based on details such as heart rate, blood pressure, steps taken today,
and temperature. The text seems to be an acknowledgement section from a document about cancer
management guidelines.

To provide a response in the requested JSON format, I would need more context or information about
the user's question. However, based on the example provided, here is a generic response:

```
{
  "recommendations": [
    {
      "type": "exercise",
      "advice": "Increase daily physical activity for better overall health."
    },
    {
      "type": "health",
      "advice": "Consult a healthcare professional for personalized advice on managing cancer and
other health conditions."
    }
  ]
}
```

Please provide more context or clarify the user's question to receive a more accurate and relevant
response.
